# DSP Series - Notebook 2: The Frequency Domain & The FFT
**Anchored in Chapters 5, 8, & 12 of *The Scientist and Engineer's Guide to DSP***

In Notebook 1, we balanced noise smoothing against time delay by guessing window sizes. In this module, we step out of the time domain and into the frequency domain using the **Fast Fourier Transform (FFT)**.

## The Marine Spectrum: Splitting the Signal from the Noise
When a vessel operates underway, its sensors capture a composite mixture of distinct physical phenomena occurring at vastly different rates (frequencies). A marine GNC engineer categorizes these into three primary bands:

1. **The Maneuvering Band (Low Frequency / "Passband"):** The massive inertia of the hull restricts physical turns, course-keeping adjustments, and zig-zag maneuvers to very low frequencies, typically below $0.03\text{ Hz}$. This is the signal we want to protect.
2. **The Wave Band (Medium Frequency / "Stopband 1"):** First-order ocean wave forces slam the hull, causing cyclical roll, pitch, and yaw telemetry spikes. Ocean swells typically occur between $0.08\text{ Hz}$ and $0.3\text{ Hz}$. 
3. **The Structural Band (High Frequency / "Stopband 2"):** Engine firing harmonics, auxiliary machinery pumps, and propeller shaft vibrations hum through the hull framing. These show up as sharp, high-frequency spikes anywhere from $5\text{ Hz}$ to $30\text{ Hz}$.

### The Mathematical Goal
Instead of viewing this data as a confusing, jagged line over time, the FFT breaks the signal down into its individual sine wave ingredients. By looking at the spectrum, we can visually identify the boundaries between these bands and mathematically justify our filter parameters:
* **Low-Pass Filter:** To pass the hull maneuver while cutting waves and engine vibrations.
* **Band-Pass Filter:** To isolate the ocean wave profile to evaluate sea state conditions.
* **High-Pass Filter:** To isolate machinery health by monitoring structural vibration changes over time.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.fft import fft, fftfreq
%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.grid'] = True

def generate_frequency_data(sea_state_amplitude, engine_rpm):
    # 60 seconds of data sampled at 100 Hz (Fs = 100)
    fs = 100.0
    t = np.linspace(0, 60, 6000, endpoint=False)
    
    # 1. True Hull Maneuver (Low frequency: 0.01 Hz and 0.02 Hz component)
    hull_maneuver = 30.0 * np.sin(2 * np.pi * 0.01 * t) + 10.0 * np.cos(2 * np.pi * 0.02 * t)
    
    # 2. Ocean Wave Disturbance (Medium frequency swell: ~0.12 Hz)
    wave_frequency = 0.12
    ocean_waves = sea_state_amplitude * np.sin(2 * np.pi * wave_frequency * t)
    
    # 3. Structural Engine Noise (High frequency: dependent on RPM)
    # E.g., 600 RPM = 10 Hz shaft frequency
    engine_hz = engine_rpm / 60.0
    engine_vibration = 1.5 * np.sin(2 * np.pi * engine_hz * t) + 0.5 * np.random.randn(len(t))
    
    # Composite Telemetry Stream
    composite_signal = hull_maneuver + ocean_waves + engine_vibration
    
    # --- Compute the FFT (Smith Chapter 12) ---
    N = len(t)
    yf = fft(composite_signal)
    xf = fftfreq(N, 1/fs)
    
    # Take positive frequencies and normalize amplitude
    frequencies = xf[:N//2]
    amplitudes = (2.0/N) * np.abs(yf[:N//2])
    
    # Plotting the Time and Frequency Domains side-by-side
    fig, axs = plt.subplots(2, 1, figsize=(11, 8))
    
    # Time Domain Plot
    axs[0].plot(t, composite_signal, 'r', alpha=0.6, label='Noisy Composite Telemetry')
    axs[0].plot(t, hull_maneuver, 'k--', linewidth=2, label='True Hull Path')
    axs[0].set_title("Time Domain: Raw Sensor Reading Array")
    axs[0].set_ylabel("Telemetry Angle [deg]")
    axs[0].set_xlabel("Time [s]")
    axs[0].legend(loc='upper right')
    
    # Frequency Domain Plot (Spectrum)
    axs[1].plot(frequencies, amplitudes, 'b-', linewidth=2)
    axs[1].set_title("Frequency Domain: The Normalized FFT Power Spectrum")
    axs[1].set_ylabel("Amplitude Peak Strength")
    axs[1].set_xlabel("Frequency [Hz]")
    
    # Dynamic annotation lines based on user controls
    axs[1].axvline(0.04, color='g', linestyle=':', linewidth=2, label='Maneuvering Boundary (~0.04 Hz)')
    axs[1].axvline(0.40, color='orange', linestyle=':', linewidth=2, label='Wave Boundary (~0.40 Hz)')
    axs[1].legend(loc='upper right')
    
    # Zoom into the lower frequency spectra area of interest (0 to 15 Hz) to see parameters clearly
    axs[1].set_xlim(0, 15)
    
    plt.tight_layout()
    plt.show()

widgets.interact(generate_frequency_data, 
                 sea_state_amplitude=widgets.FloatSlider(value=5.0, min=0.0, max=15.0, step=0.5, description='Sea State:'),
                 engine_rpm=widgets.IntSlider(value=480, min=300, max=900, step=60, description='Engine RPM:'))

interactive(children=(FloatSlider(value=5.0, description='Sea State:', max=15.0, step=0.5), IntSlider(value=48…

<function __main__.generate_frequency_data(sea_state_amplitude, engine_rpm)>

## Section 2: Drawing the Filter Boundaries

Look closely at the FFT spectrum generated above. The continuous time-domain graph looks like a chaotic jumble, but the frequency domain cleanly separates the chaos into three standalone "energy islands."



Using this spectrum, we can precisely locate the mathematical boundaries and establish parameters for our filtering sub-routines:

### 1. Setting Up a Low-Pass Filter (The Autopilot Guard)
* **The Goal:** We want to shield the autopilot loop from both wave motion and machinery hum.
* **The Analysis:** The spectrum shows that true hull motion drops off to zero by $0.03\text{ Hz}$, while ocean waves don't begin rising until $0.08\text{ Hz}$. 
* **The Parameter:** We set our **Low-Pass Cutoff Frequency ($f_c$)** right in that clear valley, around **$0.05\text{ Hz}$**. Everything above this line is suppressed, ensuring the rudder only responds to real maneuvering commands.

### 2. Setting Up a Band-Pass Filter (The Sea-State Estimator)
* **The Goal:** Suppose your vessel needs to calculate current wave conditions to determine safe cruising speeds or launch operational boats. We want to completely eliminate the ship's own path and engine noise to look *only* at the waves.
* **The Analysis:** The wave energy forms a distinct mound between $0.08\text{ Hz}$ and $0.35\text{ Hz}$.
* **The Parameter:** We set up a **Band-Pass Filter** with a lower cutoff at **$0.06\text{ Hz}$** and an upper cutoff at **$0.40\text{ Hz}$**. This slices out a clean window of pure wave telemetry, allowing us to compute significant wave height in real time.

### 3. Setting Up a High-Pass Filter (Machinery Health Diagnostics)
* **The Goal:** We want to look strictly at structural engine vibration to diagnose if a cylinder is misfiring or a bearing is failing, ignoring the slow rolling of the waves and heading changes.
* **The Analysis:** Mechanical vibration energy sits way up the spectrum. If you shift the **Engine RPM** slider, you will watch that specific high-frequency amplitude spike dance left and right along the frequency axis.
* **The Parameter:** We set a **High-Pass Cutoff Frequency ($f_c$)** at **$1.0\text{ Hz}$**. This blocks out all macro ship and environmental motion, leaving behind a clean stream of structural mechanical telemetry for predictive maintenance logs.

In [2]:
def apply_spectral_filter(filter_type):
    # Re-generate our standard 60-second composite marine dataset (Fs = 100 Hz)
    fs = 100.0
    t = np.linspace(0, 60, 6000, endpoint=False)
    N = len(t)
    
    # Components
    hull_maneuver = 30.0 * np.sin(2 * np.pi * 0.01 * t) + 10.0 * np.cos(2 * np.pi * 0.02 * t)
    ocean_waves = 6.0 * np.sin(2 * np.pi * 0.12 * t)
    engine_vibration = 2.0 * np.sin(2 * np.pi * 8.0 * t) + 0.5 * np.random.randn(len(t))
    
    composite_signal = hull_maneuver + ocean_waves + engine_vibration
    
    # 1. Forward FFT to transform signal to the Frequency Domain
    yf = fft(composite_signal)
    xf = fftfreq(N, 1/fs)
    
    # Create a copy of the spectrum to manipulate
    yf_filtered = yf.copy()
    
    # 2. Apply "Brick-Wall" Filter Parameters based on our analysis boundaries
    if filter_type == 'Low-Pass (Autopilot Mode)':
        # Pass everything below 0.05 Hz, drop everything else to zero
        cutoff = 0.05
        yf_filtered[np.abs(xf) > cutoff] = 0.0
        title_text = "Low-Pass Filter: Isolating True Hull Maneuver (< 0.05 Hz)"
        ideal_target = hull_maneuver
        
    elif filter_type == 'Band-Pass (Sea-State Mode)':
        # Pass only between 0.06 Hz and 0.40 Hz
        low_cutoff = 0.06
        high_cutoff = 0.40
        yf_filtered[(np.abs(xf) < low_cutoff) | (np.abs(xf) > high_cutoff)] = 0.0
        title_text = "Band-Pass Filter: Isolating Ocean Wave Profile (0.06 Hz - 0.40 Hz)"
        ideal_target = ocean_waves
        
    elif filter_type == 'High-Pass (Vibration Diagnostic Mode)':
        # Pass everything above 1.0 Hz
        cutoff = 1.0
        yf_filtered[np.abs(xf) < cutoff] = 0.0
        title_text = "High-Pass Filter: Isolating Structural Engine Vibration (> 1.0 Hz)"
        ideal_target = engine_vibration

    # 3. Inverse FFT (IFFT) to bring the filtered data back to the Time Domain
    filtered_signal = np.real(np.fft.ifft(yf_filtered))
    
    # Recalculate the filtered spectrum for display
    frequencies = xf[:N//2]
    original_amplitudes = (2.0/N) * np.abs(yf[:N//2])
    filtered_amplitudes = (2.0/N) * np.abs(yf_filtered[:N//2])
    
    # Plotting results
    fig, axs = plt.subplots(2, 1, figsize=(11, 8))
    
    # Time Domain Comparison
    axs[0].plot(t, composite_signal, 'r', alpha=0.15, label='Original Noisy Telemetry')
    axs[0].plot(t, ideal_target, 'k--', alpha=0.7, label='Target Mathematical Profile')
    axs[0].plot(t, filtered_signal, 'g-', linewidth=2, label='Filtered Output')
    axs[0].set_title(f"Time Domain: {title_text}")
    axs[0].set_ylabel("Angle [deg]")
    axs[0].set_xlabel("Time [s]")
    axs[0].legend(loc='upper right')
    
    # Frequency Domain Verification
    axs[1].plot(frequencies, original_amplitudes, 'r', alpha=0.2, label='Original Spectrum')
    axs[1].plot(frequencies, filtered_amplitudes, 'g-', linewidth=2, label='Filtered Spectrum')
    axs[1].set_title("Frequency Domain Spectrum Verification")
    axs[1].set_ylabel("Amplitude Strength")
    axs[1].set_xlabel("Frequency [Hz]")
    axs[1].set_xlim(0, 15)
    axs[1].legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

# Dropdown menu to let students alternate between the distinct filter configurations
widgets.interact(apply_spectral_filter, 
                 filter_type=widgets.Dropdown(
                     options=['Low-Pass (Autopilot Mode)', 'Band-Pass (Sea-State Mode)', 'High-Pass (Vibration Diagnostic Mode)'],
                     value='Low-Pass (Autopilot Mode)',
                     description='Filter Routine:'
                 ))

interactive(children=(Dropdown(description='Filter Routine:', options=('Low-Pass (Autopilot Mode)', 'Band-Pass…

<function __main__.apply_spectral_filter(filter_type)>

## Section 4: The Spectral Audit — Verifying Filter Performance

To truly master filter design as a systems architect, you must never trust a time-domain line blindly. You must audit your filtered signals in the frequency domain to verify that your filter is doing exactly what you designed it to do.

By passing our filtered Butterworth and Chebyshev signals back through the FFT, we can explicitly visualize their mathematical trade-offs:

### 1. Auditing the Butterworth Spectrum
When you inspect the filtered Butterworth spectrum, you will see a beautifully smooth, unwarped profile at the ultra-low frequencies ($< 0.05\text{ Hz}$). This confirms its **maximally flat** design philosophy. The energy peaks of the hull's true maneuver match the original signal perfectly in amplitude. As the frequency climbs toward the cutoff, the power drops away smoothly and predictably.

### 2. Auditing the Chebyshev Spectrum
When you look at the filtered Chebyshev spectrum, the penalty of its aggressive design becomes immediately visible. In the passband, the amplitude line will physically wave up and down. This is the **passband ripple** in action—the filter is mathematically exaggerating some low-frequency components while suppressing others, distorting your raw telemetry. 

However, look at how fast the Chebyshev spectrum plummets to zero right after the cutoff frequency compared to the Butterworth. It cuts through the ocean wave noise like a surgical knife.

Let's look at the final interactive code block of Notebook 2 to see this spectral showdown firsthand.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.fft import fft, fftfreq
from scipy.signal import butter, cheby1, lfilter  # <--- Added missing dependencies here
%matplotlib inline

def audit_recursive_spectra(filter_order, cutoff_hz, ripple_db):
    fs = 100.0  # 100 Hz sampling rate
    t = np.linspace(0, 60, 6000, endpoint=False) # 60 seconds for clean FFT resolution
    N = len(t)
    
    # Generate our standard composite marine signal
    hull_maneuver = 30.0 * np.sin(2 * np.pi * 0.01 * t) + 10.0 * np.cos(2 * np.pi * 0.02 * t)
    ocean_waves = 6.0 * np.sin(2 * np.pi * 0.12 * t)
    engine_vibration = 2.0 * np.sin(2 * np.pi * 8.0 * t) + 0.5 * np.random.randn(len(t))
    composite_signal = hull_maneuver + ocean_waves + engine_vibration
    
    # Design Filters
    nyq = 0.5 * fs
    normal_cutoff = cutoff_hz / nyq
    b_butt, a_butt = butter(filter_order, normal_cutoff, btype='low', analog=False)
    b_cheb, a_cheb = cheby1(filter_order, ripple_db, normal_cutoff, btype='low', analog=False)
    
    # Run Filters (Causal)
    butterworth_output = lfilter(b_butt, a_butt, composite_signal)
    chebyshev_output = lfilter(b_cheb, a_cheb, composite_signal)
    
    # Compute FFTs
    xf = fftfreq(N, 1/fs)
    frequencies = xf[:N//2]
    
    orig_amp = (2.0/N) * np.abs(fft(composite_signal)[:N//2])
    butt_amp = (2.0/N) * np.abs(fft(butterworth_output)[:N//2])
    cheb_amp = (2.0/N) * np.abs(fft(chebyshev_output)[:N//2])
    
    # Plotting the Spectrums Head-to-Head
    fig, axs = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
    
    # Top Plot: Full Spectrum Overview
    axs[0].plot(frequencies, orig_amp, 'r', alpha=0.15, label='Original Noisy Spectrum')
    axs[0].plot(frequencies, butt_amp, 'b-', linewidth=2, label='Filtered Butterworth')
    axs[0].plot(frequencies, cheb_amp, 'm-', linewidth=2, label='Filtered Chebyshev')
    axs[0].set_title("Frequency Domain: Filter Performance Overview")
    axs[0].set_ylabel("Amplitude Strength")
    axs[0].set_xlim(0, 15) # Show out to engine noise
    axs[0].legend(loc='upper right')
    
    # Bottom Plot: Zoomed Passband View (0 to 0.5 Hz) to catch the Ripple and Roll-off
    axs[1].plot(frequencies, orig_amp, 'r', alpha=0.15, label='Original Noisy Spectrum')
    axs[1].plot(frequencies, butt_amp, 'b-', linewidth=2.5, label='Filtered Butterworth')
    axs[1].plot(frequencies, cheb_amp, 'm-', linewidth=2, label='Filtered Chebyshev')
    axs[1].axvline(cutoff_hz, color='k', linestyle=':', label='Your Cutoff Target')
    axs[1].set_title("Close-up Passband Inspection: Note the Chebyshev Ripple & Roll-off Speed")
    axs[1].set_ylabel("Amplitude Strength")
    axs[1].set_xlabel("Frequency [Hz]")
    axs[1].set_xlim(0, 0.4) # Zoom deep into the maneuvering and wave zones
    axs[1].legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

widgets.interact(audit_recursive_spectra,
                 filter_order=widgets.IntSlider(value=2, min=1, max=6, description='Filter Order:'),
                 cutoff_hz=widgets.FloatSlider(value=0.06, min=0.03, max=0.2, step=0.01, description='Cutoff (Hz):'),
                 ripple_db=widgets.FloatSlider(value=1.5, min=0.2, max=3.0, step=0.1, description='Cheby Ripple (dB):'))

interactive(children=(IntSlider(value=2, description='Filter Order:', max=6, min=1), FloatSlider(value=0.06, d…

<function __main__.audit_recursive_spectra(filter_order, cutoff_hz, ripple_db)>

## Section 6: Real-World FFT Quirks — Spectral Leakage & Resolution Bins

Up to this point, our simulated signals have looked incredibly clean in the frequency domain. However, when processing live telemetry streamed from an active vessel, the raw FFT will lie to you if you do not account for two foundational signal processing constraints: **Spectral Leakage** and **Frequency Resolution Bins**.

### 1. Spectral Leakage & The Need for Windowing
The Discrete Fourier Transform (DFT/FFT) fundamentally assumes that the snippet of time-domain data you feed it is **perfectly periodic**. It assumes that if you duplicated your 60-second clip and stitched it end-to-end infinitely, the incoming waves would join together seamlessly.

In reality, a marine telemetry stream almost never starts and ends at the exact same phase of a wave cycle. When the FFT artificially loops the data, it encounters a sharp, unnatural step-discontinuity at the boundary. 



* **The Problem:** In the time domain, a sharp step-change requires infinite frequencies to construct. The FFT accommodates this by "leaking" energy across the entire spectrum. A sharp engine vibration peak at $8.35\text{ Hz}$ won't look like a clean spike; it will bleed sideways, creating wide, blurry "skirts" that can completely swallow up smaller adjacent signals.
* **The Engineering Solution:** Before running the FFT, we multiply the raw time-domain array by a **Windowing Function** (such as a *Hann* or *Hamming* window). These mathematical functions smoothly taper the edges of our data clip down to zero at the start and end. This completely eliminates the boundary step-discontinuities, forcing the spectral energy to snap back into sharp, distinct peaks.

---

### 2. The Frequency Resolution Bin Trap
Students frequently assume that if their IMU samples data at a high rate ($F_s = 100\text{ Hz}$), they can view fine frequency details down to $0.01\text{ Hz}$. This is a dangerous misconception. 

Your frequency resolution—the literal width of the spacing between individual points (bins) on your FFT plot—is governed strictly by the **total duration ($T$) of the continuous data clip**, not how fast you sample.

$$\Delta f = \frac{1}{T} = \frac{F_s}{N}$$

Where $N$ is the total number of samples in the array.

* **The GNC Consequence:** Suppose you take a short **10-second** sample of ship data ($T = 10\text{ s}$). Your frequency resolution is $\Delta f = \frac{1}{10} = 0.1\text{ Hz}$. 
* Look back at our marine spectrum: the true physical maneuvering path of a heavy ship hull happens down at **$0.01\text{ Hz}$** and **$0.02\text{ Hz}$**. 
* Because your very first analytical FFT bin after $0\text{ Hz}$ (DC offset) sits way out at $0.1\text{ Hz}$, **your true vehicle path is completely invisible.** It gets swallowed into the center line. 

To separate a slow hull turn ($0.01\text{ Hz}$) from a fast wave swell ($0.12\text{ Hz}$), you mathematically *must* record a continuous data window of at least $200\text{ seconds}$ ($T = \frac{1}{0.005}$) to give the FFT enough resolving power.

Let's execute the interactive cell below to see how spectral leakage blurs your diagnostics, and how a Hann window sharpens our vision.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from scipy.fft import fft, fftfreq
%matplotlib inline

def demonstrate_leakage_and_bins(target_freq, duration_seconds, apply_hann_window):
    fs = 100.0  # 100 Hz Sampling rate
    t = np.linspace(0, duration_seconds, int(fs * duration_seconds), endpoint=False)
    N = len(t)
    
    # Generate a pure wave signal at a non-integer bin frequency to force leakage
    # Try 5.35 Hz vs 5.00 Hz
    wave_signal = 5.0 * np.sin(2 * np.pi * target_freq * t)
    
    # Calculate Frequency Resolution Bin Size
    delta_f = 1.0 / duration_seconds
    
    if apply_hann_window:
        # Apply a Hann window taper to eliminate edge discontinuities
        processed_signal = wave_signal * np.hanning(N)
        plot_title = f"FFT Spectrum WITH Hann Windowing (Resolution Bin Δf = {delta_f:.3f} Hz)"
    else:
        # Raw unwindowed signal (Rectangular window)
        processed_signal = wave_signal
        plot_title = f"FFT Spectrum WITHOUT Windowing: Note Spectral Leakage Skirts (Δf = {delta_f:.3f} Hz)"
        
    # Compute FFT
    yf = fft(processed_signal)
    xf = fftfreq(N, 1/fs)
    
    frequencies = xf[:N//2]
    amplitudes = (2.0/N) * np.abs(yf[:N//2])
    
    # Plotting
    fig, axs = plt.subplots(2, 1, figsize=(11, 7))
    
    # Top Plot: Time Domain showing boundaries
    axs[0].plot(t, wave_signal, 'b-', label='Original Signal')
    if apply_hann_window:
        axs[0].plot(t, processed_signal, 'g-', linewidth=2, label='Windowed Tapered Signal')
    axs[0].set_title("Time Domain Representation")
    axs[0].set_ylabel("Amplitude")
    axs[0].set_xlabel("Time [s]")
    axs[0].legend(loc='upper right')
    
    # Bottom Plot: Frequency Domain showing Bins & Leakage
    axs[1].stem(frequencies, amplitudes, linefmt='g-', markerfmt='go', basefmt='r-', label='FFT Discrete Bins')
    axs[1].plot(frequencies, amplitudes, 'r-', alpha=0.3, label='Spectral Envelope')
    axs[1].set_title(plot_title)
    axs[1].set_ylabel("Measured Amplitude Strength")
    axs[1].set_xlabel("Frequency [Hz]")
    
    # Zoom close to the target frequency to clearly inspect individual resolution bins
    axs[1].set_xlim(target_freq - 2.0, target_freq + 2.0)
    axs[1].legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

widgets.interact(demonstrate_leakage_and_bins,
                 target_freq=widgets.FloatSlider(value=5.35, min=4.0, max=6.0, step=0.05, description='Target Hz:'),
                 duration_seconds=widgets.IntSlider(value=10, min=3, max=60, step=1, description='Duration (s):'),
                 apply_hann_window=widgets.Checkbox(value=False, description='Apply Hann Window'))

interactive(children=(FloatSlider(value=5.35, description='Target Hz:', max=6.0, min=4.0, step=0.05), IntSlide…

<function __main__.demonstrate_leakage_and_bins(target_freq, duration_seconds, apply_hann_window)>

## Summary: The Spectral Engineering Blueprint

Congratulations! You have successfully migrated from basic time-domain guessing to frequency-domain signal architecture. In this notebook, you learned to:

1. **Deconstruct Chaos:** Use the FFT to split a single messy sensor stream into clear environmental, physical, and mechanical energy bands.
2. **Establish Parameters:** Quantify exact filter cutoffs based on visual spectrum valleys rather than trial-and-error guesswork.
3. **Select Your Weapon:** Weigh the flat, stable passband tracking of a **Butterworth filter** against the aggressive, sharp noise truncation of a **Chebyshev filter**.

### What's Next?
You now know how to design time-domain moving averages and frequency-domain recursive IIR filters. But what happens when your noise isn't steady or predictable? What happens when wave frequencies shift constantly as the ship moves from deep ocean swells into shallow coastal channels? 

Recursive filters have fixed parameters—they cannot adapt. In the next notebook, we will break open the ultimate crown jewel of modern GNC system estimation: **The Kalman Filter**. We will learn how to fuse our filtered IMU data with dynamic model physics to track the ship's true state with absolute certainty, even in a blinding storm.

## Looking Ahead - Notebook 3: Predictive Filtering & The Kalman Filter

You have mastered the art of frequency analysis, but you are still bound by a major real-world limitation: **static filters cannot adapt.** If your ship changes speed, or if the ocean waves shift from long, slow deep-water swells to short, fast coastal chops, your hardcoded Butterworth or Chebyshev parameters will quickly become obsolete—either cutting out real maneuvers or letting destructive noise pass right into your steering actuators.

### The Next Level: Fusing Math with Machine Physics
To solve this, a modern GNC system doesn't just look at the sensor data. It looks at the **vessel itself**. 

In **Notebook 3: State Estimation & The Kalman Filter**, we will leave classical DSP behind and step into optimal estimation theory. We will build a live, mathematical model of a hull's hydrodynamics (mass, inertia, and hydrodynamic drag) and run it directly alongside our sensor stream. 

By calculating a dynamic, real-time balance of uncertainty between our physical model equations and our live IMU telemetric streams, we will construct an adaptive filter that can track a ship's true vector flawlessly—even if the sensors are swimming in unpredictable, changing storm noise. 

**Get ready: in the next module, we build the brains of modern autonomous navigation.**